# Mettler Toledo MT-SICS scales

`MTSICSDriver` connects to Mettler Toledo scales and weigh modules that implement the
MT-SICS serial protocol. It discovers the commands advertised by the connected instrument
during setup, so one driver can support multiple models with different command sets.

| Property | Value |
|---|---|
| Protocol | MT-SICS (Mettler Toledo Standard Interface Command Set) |
| Communication | RS-232 serial |
| Driver default baud rate | 9600 |
| Default USB adapter match | `0x0403:0x6001` (FTDI FT232R) |
| PyLabRobot API | `MTSICSDriver` |
| Hardware-validated example | [WXS205SDU WXA-Bridge](https://www.mt.com/us/en/home/products/Industrial_Weighing_Solutions/high-precision-weigh-sensors/weigh-module-wxs205sdu-15-11121008.html), firmware 1.10 |

Support is based on the commands reported by the instrument, not on a hard-coded model name.
The WXS205SDU is the currently hardware-validated example; other MT-SICS models may expose a
different subset of the methods shown below.


```{device-card} mettler-toledo-wxs205sdu
```

## How it communicates

MT-SICS (Mettler Toledo Standard Interface Command Set) is an ASCII request/response
protocol. Commands and responses are terminated by carriage return and line feed. The driver
communicates through PyLabRobot's serial transport and handles framing, response parsing,
multi-line responses, and MT-SICS error codes.

During `setup()`, the driver resets the interface, queries `I0` to discover supported
commands, reads the device identity and firmware, and selects grams as the host unit when the
instrument supports that setting. A method whose MT-SICS command is unavailable on the
connected model raises `MettlerToledoError` before sending it.

## Physical setup

Hardware layouts and connectors vary by model. Some systems use a separate load cell,
electronic unit, and terminal, while others integrate these components. Follow the manual for
your instrument, then connect its MT-SICS-capable RS-232 interface to the computer. A
USB-to-serial adapter is normally required.

The driver defaults to the FTDI FT232R VID:PID `0x0403:0x6001`. If your adapter uses different
identifiers, pass the serial port explicitly or provide its `vid` and `pid` when creating the
driver.

Install PyLabRobot with serial support before continuing:

```bash
pip install "pylabrobot[serial]"
```

```{warning}
Place the scale on a stable, level surface and follow the warm-up and environmental guidance
for your model before measuring. An instrument that is not ready may report that a command is
understood but not currently executable.
```

## Connect

Create the driver with the serial port used by the scale and call `setup()`. Port names are
typically `/dev/ttyUSB0` on Linux, `/dev/cu.usbserial-*` on macOS, and `COM3` or similar on
Windows. If `port` is omitted, the driver searches for the configured USB VID and PID.


In [ ]:
from pylabrobot.mettler_toledo import MTSICSDriver

scale = MTSICSDriver(port="/dev/cu.usbserial-110")  # replace with your serial port
await scale.setup()

### Confirm the discovered instrument

`setup()` records the identity and capacity reported by the instrument. Inspect these values
before starting a measurement workflow, especially when several serial devices are connected.


In [ ]:
print(f"Model: {scale.device_type}")
print(f"Serial number: {scale.serial_number}")
print(f"Firmware: {scale.firmware_version}")
print(f"Capacity: {scale.capacity} g")

## Zero the empty scale

Remove everything from the weighing platform, then call `zero()`. The default waits for a
stable reading. Use `zero(timeout=0)` only when an immediate zero is preferable to waiting for
stability.

In [ ]:
await scale.zero()

## Tare a container

Place the empty container on the platform and wait for the reading to settle. `tare()` stores
its weight so subsequent readings report only the sample's net weight.

In [ ]:
await scale.tare()

## Read a stable weight

Add the sample to the tared container. `read_weight()` waits for stability and returns the
weight in grams as a `float`.

In [ ]:
weight_g = await scale.read_weight()
print(f"Weight: {weight_g:.4f} g")

### Read immediately

Use `timeout=0` when the current value is needed even if it is still changing. The result is
still expressed in grams.

In [ ]:
current_weight_g = await scale.read_weight(timeout=0)
print(f"Current weight: {current_weight_g:.4f} g")

### Inspect the stored tare

Query the tare value currently stored by the scale.

In [ ]:
tare_weight_g = await scale.request_tare_weight()
print(f"Stored tare: {tare_weight_g:.4f} g")

### Clear the tare

Remove the container, then clear the stored tare when the workflow is finished.

In [ ]:
await scale.clear_tare()

## Measure the internal temperature

Some MT-SICS instruments, including the hardware-validated WXS205SDU, expose an internal
temperature sensor with the `M28` command. This can be useful when temperature affects density
calculations in gravimetric verification. Skip this call if your model does not advertise
`M28`; the driver will otherwise raise `MettlerToledoError`.


In [ ]:
temperature_c = await scale.measure_temperature()
print(f"Internal temperature: {temperature_c:.1f} °C")

## Query device identity

Identity methods can be called again after setup when a workflow needs to record instrument
provenance alongside its measurements. The additional identity fields below are model
dependent; an unsupported command raises `MettlerToledoError`.


In [ ]:
identity = {
  "serial_number": await scale.request_serial_number(),
  "device_type": await scale.request_device_type(),
  "model": await scale.request_model_designation(),
  "firmware": await scale.request_firmware_version(),
  "software_material_number": await scale.request_software_material_number(),
}
identity

## Query device status

The following read-only calls report the instrument's clock, uptime, and next configured
service date when their corresponding commands are supported by the connected model.


In [ ]:
status = {
  "date": await scale.request_date(),
  "time": await scale.request_time(),
  "uptime_minutes": await scale.request_uptime_minutes(),
  "next_service_date": await scale.request_next_service_date(),
}
status

## Read weight with MT-SICS status

`request_net_weight_with_status()` exposes the structured MT-SICS response when a workflow
needs the stability state, unit code, readability, approval state, or tare information in
addition to the numeric weight.

In [ ]:
response = await scale.request_net_weight_with_status()
print(f"Command: {response.command}")
print(f"Status: {response.status}")
print(f"Data: {response.data}")

## Read a multi-line response

Some MT-SICS commands return several lines. The driver returns them as a list of
`MettlerToledoResponse` objects. For example, instruments that implement `I14` can report
installed components through device-information category 0.


In [ ]:
device_info = await scale.request_device_info(category=0)
for line in device_info:
  print(line.command, line.status, line.data)

## Inspect weighing configuration

These configuration-query methods are read-only and model dependent. Their returned values are
MT-SICS setting codes; consult the Mettler Toledo MT-SICS reference for the meaning of each code
on your model.


In [ ]:
configuration = {
  "weighing_mode": await scale.request_weighing_mode(),
  "environment_condition": await scale.request_environment_condition(),
  "auto_zero": await scale.request_auto_zero(),
  "update_rate_hz": await scale.request_update_rate(),
}
configuration

## Command availability across models

Not every MT-SICS instrument implements every command. During `setup()`, the driver queries
`I0` and records the commands advertised by the connected instrument. Calling a method whose
command was not advertised raises `MettlerToledoError` before anything is sent.

For example, the WXS205SDU WXA-Bridge used for hardware validation does not expose the timed
zero/tare/read commands, display commands, the cancel-all command, or remaining-range query.
Those methods remain available for other MT-SICS models that advertise the corresponding
commands.

Methods such as `set_device_id()`, `set_date()`, and `set_time()` intentionally change
persistent device state and are therefore not run in this hello-world guide.


## Disconnect

Always stop the driver when the workflow finishes. `stop()` attempts to reset the interface
to a determined state before closing the serial connection.

In [ ]:
await scale.stop()